# Impact Platform — Phase 6.5 Water Training (Kaggle)

**Run this notebook in Kaggle with GPU enabled** (Settings → Accelerator → GPU T4 × 2)

**Enable Internet** in Settings (needed for dataset downloads)

This trains the **water** body detector (YOLOv8n) and change detector (ChangeFormer).

Recommended dataset: **S1S2-Water** (Hugging Face mirror) or **GLH-Water**

Expected time: ~2-3 hours on T4×2

In [ ]:
# @title 1. Setup — clone repo
import os
import subprocess

REPO_URL = "https://github.com/YOUR_ORG/impact-platform.git"  # <-- CHANGE THIS
REPO_DIR = "/kaggle/working/impact-platform"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(os.path.join(REPO_DIR, "apps", "ml-service"))
print(f"Working in: {os.getcwd()}")

In [ ]:
# @title 2. Install requirements
!pip install -q -r training/requirements_kaggle.txt

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)")

In [ ]:
# @title 3. Download water dataset (choose ONE)
# Option A: S1S2-Water via Hugging Face (fastest, ~170 GB full — use subset)
!huggingface-cli download idomogalla/s1s2_water \
    --local-dir data/raw/s1s2_water \
    --include "part1/*"  # just first shard for testing

# Option B: GLH-Water (manual — download from project page first)
# !python -m training.data.download_datasets download_glh_water

# Option C: ATLANTIS (from GitHub Google Drive)
# !python -m training.data.download_datasets download_atlantis

In [ ]:
# @title 4. Prepare YOLO format (water)
# Using S1S2-Water subset as example
!python -m training.data.prepare_yolo water --source s1s2_water

In [ ]:
# @title 5. Train YOLOv8n water body detector (~2 hrs on T4×2)
!python -m training.scripts.train_yolo water --source s1s2_water

In [ ]:
# @title 6. Train ChangeFormer for water (~1 hr)
!python -m training.scripts.train_change water --source s1s2_water

In [ ]:
# @title 7. Evaluate
!python -m training.scripts.evaluate water --source s1s2_water --output eval/water_eval_v1.0.json

In [ ]:
# @title 8. Verify weights
from pathlib import Path

weights_dir = Path("weights")
for w in ["water_yolov8n.pt", "changeformer_water.pt", "yolov8n.pt"]:
    p = weights_dir / w
    print(f"{w}: {'EXISTS' if p.exists() else 'MISSING'} ({p.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# @title 9. Promote water model registry (after uploading weights)
# Requires SUPABASE_SERVICE_KEY + SUPABASE_URL as Kaggle Secrets
# !python -m training.scripts.promote_model water --version v1.0 --source s1s2_water